# 🎬 ShouldIPost? — готовий розв'язок (end-to-end)

**Задача (ТЗ):** для ще-**не**-опублікованого TikTok-кандидата (підпис, тривалість, час,
опц. креатор) передбачити, чи варто його постити → **Post / Do not post / Unsure** з
каліброваною ймовірністю, факторами та чесною нотаткою про обмеження.

Цей ноутбук проходить **усі етапи** конвеєра з поясненнями. Він **підтягує модулі-ноутбуки `notebooks/`** через `%run`, а ключову логіку дублює інлайн, щоб
було видно, *що* і *чому* відбувається.

> ▶️ **Запускати з кореня репозиторію** `shouldipost/` у венві проєкту:
> `source .venv/bin/activate && jupyter lab notebooks/solution.ipynb`
> (або `jupyter nbconvert --execute --to notebook notebooks/solution.ipynb`).

**Карта етапів:** 0 Налаштування → 1 Постановка → 2 Дані → 3 EDA → 4 Мітка →
5 Очистка+спліт → 6 Фічі → 7 Anti-leakage → 8 Моделі+вибір → 9 Оцінка →
10 Поріг рішення → 11 Inference-демо → 12 Висновки → 13 **Що сабмітити в git**.

## Етап 0 — Налаштування

Підтягуємо модулі-ноутбуки через `%run`. `config` — єдине джерело правди (схема, leakage-allowlist,
пороги). `features` — стейтлес-трансформ фічей зі стіною проти витоку.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
# модулі підтягуються з ноутбуків через %run (повністю ноутбучний проєкт)
%run 03_data_prep.ipynb
%run 04_evaluate.ipynb

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 40)
print("Фічей у контракті:", len(features.FEATURE_COLUMNS))
print("Заборонені (label-only) колонки:", config.POSTHOC_COLS)

## Етап 1 — Постановка задачі та припущення

- **Тип:** бінарна класифікація (успіх / ні) з **абстенцією** ("Unsure").
- **Що відомо ДО посту** (дозволені фічі): `description`, `duration`, `create_time`, креатор.
- **Відомо лише ПІСЛЯ** (тільки мітка, ніколи не вхід): `play_count`, `digg/share/comment/collect/repost_count`.
- **Метрика під рішення:** слот для посту дефіцитний → цінуємо **калібрування** (Brier) і
  **precision класу Post**; ROC/PR-AUC для ранжування.
- **Чесність:** на слабкому сигналі краще абстейнити, ніж вдавати впевненість.

🧠 Головне правило: **post-hoc метрики — це мітка, а не фіча.** Усе інше випливає з нього.

## Етап 2 — Завантаження даних

`data_prep.load_raw()` завантажить `train.csv` з HuggingFace (`datahiveai/Tiktok-Videos`,
**CC BY-NC 4.0** — лише дослідницьке використання), якщо його ще немає локально.

In [ ]:
raw = data_prep.load_raw()
print("розмір:", raw.shape)
print("колонки:", list(raw.columns))
raw[["author_unique_id", "description", "duration", "create_time", "play_count"]].head(3)

## Етап 3 — EDA: підтверджуємо hazard'и ЧИСЛАМИ

Не віримо опису наосліп — перевіряємо. Ключова знахідка, що визначить дизайн мітки.

In [ ]:
print("Креаторів усього:", raw["author_unique_id"].nunique())          # лише 4!
display(raw["author_unique_id"].value_counts())
print("repost_count усі нулі:", bool((raw["repost_count"].fillna(0) == 0).all()))
ct = pd.to_datetime(raw["create_time"], unit="s", utc=True)
print("Діапазон дат:", ct.min().date(), "→", ct.max().date(), "(дрейф у часі)")
print("Тривалість: медіана", raw["duration"].median(), "| нулів", int((raw["duration"]==0).sum()),
      "| null", int(raw["duration"].isna().sum()))

In [ ]:
# Концентрація креаторів + важкий хвіст переглядів (логарифмічна шкала)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
raw["author_unique_id"].value_counts().plot.bar(ax=ax[0], color="#FE2C55",
                                                 title="Відео на креатора (їх лише 4)")
np.log10(raw["play_count"].replace(0, np.nan).dropna()).hist(bins=40, ax=ax[1], color="#25F4EE")
ax[1].set_title("log10(play_count)"); ax[1].set_xlabel("log10(переглядів)")
plt.tight_layout(); plt.show()

**Висновок EDA:** даними керують **4 мега-креатори**. Абсолютна мітка переглядів міряла б
*славу*, не якість. Тому беремо **within-creator** мітку (наступний етап).

## Етап 4 — Визначення мітки (label C, within-creator)

`success = engagement_rate > медіана_ER_цього_креатора`, де
`ER = (digg+share+comment+collect)/play_count` (repost виключено — всі нулі).
⚠️ Пороги медіан рахуємо **лише на train** і **заморожуємо** (інакше витік).

In [ ]:
# спершу очистка (дроп битих рядків + мертвих колонок), тоді ТЕМПОРАЛЬНИЙ спліт
clean = data_prep.clean(raw)
train_df, test_df = data_prep.temporal_split(clean)            # старе→train, нове→test
thr = labels.fit_creator_thresholds(train_df)                  # пороги ЛИШЕ з train
print("Медіани ER креаторів (train):", {k: round(v, 3) for k, v in thr["per_creator"].items()})

y_tr_full = labels.make_labels(train_df, thr)
y_te_full = labels.make_labels(test_df, thr)                   # ті самі заморожені пороги
print(f"pos rate: train={y_tr_full.mean():.3f}  test={y_te_full.mean():.3f}  <-- ДРЕЙФ!")

Бачимо **дрейф**: train ≈50% позитивних (медіана ділить навпіл), а на найновіших постах —
значно менше. Темпоральний спліт чесно це викрив. Перевіримо вибір мітки демонстрацією A-vs-C
далі.

## Етап 5 — Очистка + темпоральний спліт (деталі)

Спліт ріжемо за **значенням часу** (а не позицією), щоб однакові мітки часу не потрапили по
обидва боки. Усе подальше підганяється **лише на train**.

In [ ]:
print(f"train: {len(train_df)} рядків | test: {len(test_df)} рядків")
ct_tr = pd.to_datetime(train_df["create_time"], unit="s", utc=True)
ct_te = pd.to_datetime(test_df["create_time"], unit="s", utc=True)
ct_tr = ct_tr[ct_tr.dt.year > 2010]; ct_te = ct_te[ct_te.dt.year > 2010]
print("train дати:", ct_tr.min().date(), "→", ct_tr.max().date())
print("test  дати:", ct_te.min().date(), "→", ct_te.max().date(), "(строго новіші)")

## Етап 6 — Інженерія фічей (лише pre-post)

`features.engineer_features` — стейтлес: той самий код на train та на inference. Текстові
фічі вінзоризовані, час закодований циклічно (sin/cos), креатор у фічі **не входить**
(мітка вже прибрала його рівень).

In [ ]:
def make_xy(df, thr):
    y = labels.make_labels(df, thr); m = y.notna()
    d = df.loc[m].reset_index(drop=True)
    X = features.engineer_features(d)                 # стіна проти витоку — всередині
    return X, y.loc[m].reset_index(drop=True).astype(int), d

Xtr, ytr, train_df2 = make_xy(train_df, thr)
Xte, yte, test_df2  = make_xy(test_df, thr)
print("Матриця фічей:", Xtr.shape, "| колонки:")
print(features.FEATURE_COLUMNS)
Xtr.head(3)

## Етап 7 — Anti-leakage: перевірка стіни

Фічі **фізично** не можуть містити post-hoc метрик. Guard кидає помилку, якщо хтось
спробує протягнути `play_count` у фічі.

In [ ]:
features.assert_no_leakage(Xtr)                         # ок — чисто
print("OK: у фічах немає post-hoc/ідентифікаторів")
try:
    bad = Xtr.copy(); bad["play_count"] = 1
    features.assert_no_leakage(bad)
except AssertionError as e:
    print("✋ guard спрацював:", e)

## Етап 8 — Baselines + моделі + автовибір

Три baseline (більшість, історія креатора, проста LR на 3 фічах) + повна LogReg +
калібрований HistGradientBoosting. Деплой-кандидата обираємо за **OOF-Brier на train**
(калібрування = суть продукту), а не на тесті.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import brier_score_loss

def make_logreg():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000, C=0.5, random_state=42))])

def make_hgb():
    base = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05,
                                          l2_regularization=1.0, min_samples_leaf=30,
                                          random_state=42)
    return CalibratedClassifierCV(base, method="sigmoid", cv=5)

probs = {}
probs["majority"] = DummyClassifier(strategy="prior").fit(Xtr, ytr).predict_proba(Xte)[:, 1]
rate = ytr.groupby(train_df2["author_unique_id"].values).mean()         # історія креатора
probs["creator_hist"] = test_df2["author_unique_id"].map(rate).fillna(ytr.mean()).to_numpy()

cv = StratifiedKFold(5, shuffle=True, random_state=42)
oof = {}
for name, mk in [("logreg", make_logreg), ("hgb", make_hgb)]:
    oof[name] = cross_val_predict(mk(), Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
    probs[name] = mk().fit(Xtr, ytr).predict_proba(Xte)[:, 1]

oof_brier = {k: round(brier_score_loss(ytr, v), 3) for k, v in oof.items()}
deployed = min(oof_brier, key=oof_brier.get)
print("OOF Brier (менше = краще калібр.):", oof_brier, "→ ДЕПЛОЙ:", deployed)

## Етап 9 — Оцінка: head-to-head + калібрування + сигнал без дрейфу

Метрики під рішення: ROC/PR-AUC, **Brier**, accuracy. Плюс калібрувальна крива і
random-CV (показує, що сигнал є, але його маскує дрейф).

In [ ]:
res = {n: evaluate.binary_metrics(yte, p, t_high=0.5) for n, p in probs.items()}
tbl = pd.DataFrame(res).T[["roc_auc", "pr_auc", "brier", "accuracy"]].round(3)
print(tbl)
print("\nbase rate тесту:", round(yte.mean(), 3),
      "| Brier-підлога (константа base rate):", round(brier_score_loss(yte, np.full(len(yte), yte.mean())), 3))

In [ ]:
from sklearn.calibration import calibration_curve
plt.figure(figsize=(5, 5)); plt.plot([0, 1], [0, 1], "--", color="gray", label="ідеал")
for n in ["logreg", "hgb"]:
    fp, mp = calibration_curve(yte, probs[n], n_bins=8, strategy="quantile")
    plt.plot(mp, fp, "o-", label=f"{n} (Brier={brier_score_loss(yte, probs[n]):.3f})")
plt.xlabel("середня передбачена p"); plt.ylabel("частка позитивних")
plt.title("Калібрування (temporal test)"); plt.legend(); plt.show()

In [ ]:
# random-CV: пороги мітки перераховуються ПОФОЛДОВО (leakage-clean) -> "чи є сигнал?"
from sklearn.model_selection import KFold
def cv_auc(df):
    Xall = features.engineer_features(df); idx = np.arange(len(df)); aucs = []
    for tr, va in KFold(5, shuffle=True, random_state=42).split(idx):
        th = labels.fit_creator_thresholds(df.iloc[tr]); y = labels.make_labels(df, th)
        ot = tr[y.iloc[tr].notna().values]; ov = va[y.iloc[va].notna().values]
        m = HistGradientBoostingClassifier(max_depth=3, max_iter=200, min_samples_leaf=30, random_state=42)
        m.fit(Xall.iloc[ot], y.iloc[ot].astype(int))
        aucs.append(evaluate.roc_auc_score(y.iloc[ov].astype(int), m.predict_proba(Xall.iloc[ov])[:, 1]))
    return float(np.mean(aucs))

print("Temporal-test AUC (з дрейфом):", round(evaluate.roc_auc_score(yte, probs[deployed]), 3))
print("Random-CV AUC (без дрейфу)  :", round(cv_auc(clean), 3), " <- сигнал є, дрейф маскує")
demo = evaluate.label_a_vs_c_demo(clean)
print("Демо A-vs-C: creator-only AUC під A =", round(demo["creator_only_auc_label_A"], 2),
      "| під C =", round(demo["creator_only_auc_label_C"], 2), "(славу прибрано з цілі)")

## Етап 10 — Поріг рішення + смуга абстенції

Пороги тюнимо на **train OOF** (precision-таргет + floor на support), із мінімальним
довірчим відступом, щоб "Unsure" був змістовним.

In [ ]:
(t_low, t_high), reached = evaluate.choose_thresholds(ytr, oof[deployed])
dep = evaluate.binary_metrics(yte, probs[deployed], t_high=t_high, t_low=t_low)
print(f"Смуга: skip ≤ {t_low:.2f} | Unsure | post ≥ {t_high:.2f}  (precision-target досягнуто: {reached})")
print(f"Покриття на тесті — Post: {dep['frac_post']:.0%} | Unsure: {dep['frac_unsure']:.0%} | "
      f"Do-not-post: {dep['frac_dont']:.0%}")
print("Confusion [[TN,FP],[FN,TP]]:", dep["confusion_matrix"])

## Етап 11 — Inference-демо (через `06_inference.ipynb`)

`inference.recommend()` приймає **лише pre-post** вхід, ніколи не падає, повертає
рекомендацію + ймовірність + фактори + схожі відео + нотатку. Використовує збережений
артефакт `models/model.joblib` (за потреби натренуємо).

In [ ]:
%run 06_inference.ipynb
if not inference.is_trained():
    print('Спершу запусти 05_train.ipynb, щоб зберегти models/model.joblib')
demo_cases = [
    dict(caption="POV: коли трюк нарешті вдався 🔥 #fyp #magic", duration=18,
         when="2025-03-01T19:30:00", creator="zachking"),
    dict(caption="", duration=None, when=None),                       # порожній/неповний вхід
    dict(caption="FOLLOW ME!!! LINK IN BIO!!! 🔥🔥🔥", duration=12, when="2025-02-01T20:00:00"),
]
for kw in demo_cases:
    r = inference.recommend(**kw)
    print(f"\n• {kw.get('caption','')[:40]!r}")
    print(f"  → {r['recommendation']} (p={r['probability']}) — {r['decision_reason']}")
    print("  топ-фактори:", [(f['label'], f['direction']) for f in r['factors'][:3]])
    if r["warnings"]: print("  ⚠️", r["warnings"][0])

## Етап 12 — Висновки та обмеження (чесно)

- На **дрейфованому** тесті ранжування ≈ випадкове (ROC-AUC CI включає 0.5; PR-AUC = base rate).
- Але модель **найкраще калібрована** з кандидатів і **багато абстейнить** (~70% "Do not post").
- **Сигнал є** (random-CV AUC ≈0.65), його маскує дрейф; фічі без absolute-year не бачать
  секулярного спаду — тому holdout навмисно песимістичний.
- **Що покращити далі:** rolling-window медіани креаторів, drift-aware ознаки, більше
  креаторів, ембединги підписів, SHAP, моніторинг калібрування в проді.

> Цінність інструмента — **калібрований нудж + чесна абстенція + пояснення**, а не оракул
> вірусності.

## Етап 13 — Що сабмітити в git і в якій ПОСЛІДОВНОСТІ

Проєкт повністю ноутбучний (без `.py`). Раw-дані й артефакти не комітимо.

### Крок 0 — база на `main`
```bash
git init && git checkout -b main
git add .gitignore requirements.txt PLAN.md
git commit -m "Project plan and scaffolding"
```

### Крок 1 — гілка для роботи
```bash
git checkout -b implementation
```

### Крок 2 — логічні коміти В ЦЬОМУ ПОРЯДКУ
```bash
# 1) EDA
git add notebooks/eda.ipynb reports/eda_report.md reports/plots/
git commit -m "EDA notebook: confirm dataset hazards"

# 2) Ядро пайплайна (ноутбуки-модулі)
git add notebooks/00_config.ipynb notebooks/01_features.ipynb notebooks/02_labels.ipynb notebooks/03_data_prep.ipynb
git commit -m "Pipeline notebooks: config, features, label, data prep"

# 3) Тренування + оцінка
git add notebooks/04_evaluate.ipynb notebooks/05_train.ipynb reports/metrics.json reports/model_comparison.md reports/model_card.md
git commit -m "Training + evaluation notebooks"

# 4) Inference + тести + демо
git add notebooks/06_inference.ipynb notebooks/07_tests.ipynb notebooks/08_demo.ipynb
git commit -m "Inference, tests and demo notebooks"

# 5) Наскрізний ноутбук + README + навчальні матеріали
git add notebooks/solution.ipynb notebooks/README.md README.md docs/
git commit -m "Solution notebook, README, learning materials"
```

### Крок 3 — PR `implementation -> main`
```bash
gh repo create USER/shouldipost --private
git push -u origin main && git push -u origin implementation
gh pr create --base main --head implementation \
  --title "ShouldIPost? — pre-publish TikTok success predictor (notebooks)" \
  --body-file reports/PR_DESCRIPTION.md
```

### Не комітимо (`.gitignore`)
```
data/raw/  data/processed/      # сирі/проміжні дані (CC BY-NC + великі)
models/*  (крім .gitkeep)       # артефакти — відтворюються через 05_train.ipynb
.venv/  __pycache__/  .ipynb_checkpoints/  .DS_Store
```

### Опис PR має містити (за ТЗ)
1. Запуск локально. 2. Підхід. 3. Дані+ціль. 4. Результати vs baseline.
5. Обмеження + що покращити. 6. AI-інструменти / платні API / компют.

✅ Порядок: **PLAN -> EDA -> pipeline -> train/eval -> inference/tests/demo -> solution+docs -> PR.**
